# Benchmarking of Regression Models

This notebook analyzes the benchmarking results of regression models
for uroflow prediction from segmented audio recordings.

All experiments were executed offline and stored as CSV artifacts.
This notebook performs analysis only (no model training).


In [ ]:
import numpy as np
import pandas as pd



In [ ]:
df = pd.read_csv("../results/benchmark_results.csv")

print(df.shape)
df.head()


In [ ]:
assert not df.empty
assert df["MAE"].min() > 0
assert df["MAE"].max() < 15
assert set(df["model"].unique()) == {"RF", "GB", "SVR"}


In [ ]:
#Table for Figure 6(Model Comparison across Devices)
fig6_table = (
    df.groupby(["device", "model"])
      .agg(
          MAE_mean=("MAE", "mean"),
          MAE_std=("MAE", "std")
      )
      .reset_index()
)

fig6_table


In [ ]:
#Table for Figure 8(Segment size analysis)
fig8_table = (
    df.groupby(["device", "time_ms"])
      .agg(
          MAE_mean=("MAE", "mean"),
          MAE_std=("MAE", "std")
      )
      .reset_index()
)

fig8_table


In [ ]:
fig6_table.to_csv("../results/fig6_table.csv", index=False)
fig8_table.to_csv("../results/fig8_table.csv", index=False)


In [20]:
def extract_fft_features(audio_file, fmin, fmax, nbins):
    y, sr = librosa.load(audio_file, sr=None)

    fft_vals = np.abs(scipy.fft.rfft(y))
    freqs = scipy.fft.rfftfreq(len(y), d=1/sr)

    mask = (freqs >= fmin) & (freqs <= fmax)
    freqs = freqs[mask]
    fft_vals = fft_vals[mask]

    bins = np.linspace(freqs.min(), freqs.max(), nbins + 1)
    features = []

    for i in range(nbins):
        idx = (freqs >= bins[i]) & (freqs < bins[i+1])
        features.append(fft_vals[idx].sum())

    return np.array(features, dtype=np.float32)


In [21]:
def load_segmented_dataset(base_dir, fmin, fmax, nbins):
    X, y = [], []

    for patient in os.listdir(base_dir):
        patient_dir = os.path.join(base_dir, patient)
        if not os.path.isdir(patient_dir):
            continue

        labels = pd.read_csv(os.path.join(patient_dir, "labels.csv"))

        for _, row in labels.iterrows():
            audio_file = os.path.join(patient_dir, row["audio_file"])
            features = extract_fft_features(audio_file, fmin, fmax, nbins)

            X.append(features)
            y.append(row["flow"])

    return np.array(X), np.array(y)


In [6]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [7]:
import sys
import os

# Make project root importable
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.features.extract import load_segmented_dataset

BASE_DIR = "../data/processed/processed_1000ms/um"

assert os.path.exists(BASE_DIR), f"Path does not exist: {BASE_DIR}"

X, y = load_segmented_dataset(
    base_dir=BASE_DIR,
    fmin=0,
    fmax=8000,
    nbins=20
)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1306, 20)
y shape: (1306,)
